# Домашнее задание №10-11

In [ ]:
!pip -q install transformers datasets nltk sentencepiece accelerate scikit-learn

In [ ]:
# Импорт библиотек
import re
import numpy as np
import pandas as pd
import torch
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

nltk.download("punkt")
nltk.download("stopwords")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# dataset для заданий
dataset = load_dataset("ag_news")

train_df = pd.DataFrame(dataset["train"][:4000])
test_df = pd.DataFrame(dataset["test"][:1000])

label_mapping = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

train_df["label_name"] = train_df["label"].map(label_mapping)
test_df["label_name"] = test_df["label"].map(label_mapping)

train_df.head()


# Блок 1 — Classical NLP

## Задание 1 — Продвинутый preprocessing (1 балл)

Одним из первых этапов в NLP является `preprocessing` текста. Качественный preprocessing обязательно:
- удаляет шум,
- уменьшает `sparsity`,
- помогает модели обобщать сущности.

### Задание

Реализуйте функцию `advanced_preprocess`.

Требования:
1. lowercase
2. удаление URL
3. удаление чисел
4. удаление пунктуации
5. токенизация
6. удаление stopwords
7. удаление токенов длины < 3
8. вернуть список токенов


In [ ]:
stop_words = set(stopwords.words("english"))

def advanced_preprocess(text: str):
    # Your code here

    return tokens
sample = "Transformers 101 are AMAZING!!! Visit https://huggingface.co today."
tokens = advanced_preprocess(sample)
tokens

In [ ]:
assert isinstance(tokens, list)
assert "transformers" in tokens
assert "amazing" in tokens
assert all(token.isalpha() for token in tokens)
assert "are" not in tokens
assert all(len(token) >= 3 for token in tokens)
assert not any("http" in token for token in tokens)
print("All checks passed.")


## Задание 2 — TF-IDF + error analysis (2 балла)

В задачах классификации текста важно не только выбрать модель, но и правильно представить текст в числовом виде.

В этом задании вы построите классический (дедовский) NLP `pipeline`:

преобразование текста в `TF-IDF` признаки,
обучение линейного классификатора.

Используйте:

`TF-IDF` для построения признакового пространства,
`LogisticRegression` для многоклассовой классификации новостей.

Дополнительно используйте:

* биграммы `(ngram_range=(1, 2))`, чтобы учитывать короткие словосочетания
* `sublinear_tf=True`, чтобы уменьшить влияние слишком частых токенов

### Часть A

Создайте pipeline:
- TF-IDF
- LogisticRegression

Требования:
- `max_features=5000`
- `ngram_range=(1, 2)`
- `sublinear_tf=True`

### Часть B

Обучите модель.

Дополнительно:
- вычислите `accuracy`,
- постройте `confusion matrix`,
- найдите 5 ошибочно классифицированных примеров.


In [ ]:
# dataset уже доступен (см. ячейку №3)


# Your code here

pipeline_model = None

pipeline_model.fit(
    train_df["text"],
    train_df["label"]
)

acc = None
cm = None
misclassified = None
print("Accuracy:", acc)
print(cm)

In [ ]:
assert acc > 0.84
assert cm.shape == (4, 4)
assert len(misclassified) == 5
vectorizer = pipeline_model.named_steps["tfidf"]
features = vectorizer.get_feature_names_out()
assert len(features) <= 5000
assert any(" " in feature for feature in features)
print("All checks passed.")


## Задание 3 — Semantic retrieval через TF-IDF (2 балла)

`Retrieval` — одна из базовых задач NLP.

Под retrieval обычно понимают поиск наиболее релевантных документов по пользовательскому запросу.

Даже без больших нейросетей можно строить эффективные `retrieval`-системы:

индексировать документы,
получать embedding / vector representation текста,
вычислять близость между запросом и документами,
ранжировать результаты.
### Часть A

Создайте retrieval-функцию:
```python
retrieve_top_k(query, documents, k=3)
```

### Часть B

Функция должна:
- строить TF-IDF,
- считать `cosine similarity`,
- возвращать top-k документов и score.

**Обратите внимание**:

* более релевантные документы должны иметь больший `similarity score`,
* результаты должны быть отсортированы по убыванию `similarity`.


In [ ]:
documents = [
    "Transformers are widely used in NLP.",
    "PyTorch is popular for deep learning.",
    "Football is a popular sport.",
    "Large language models use attention mechanisms.",
    "Neural networks can process text."
]

query = "attention in language models"

def retrieve_top_k(query, documents, k=3):
    # Your code here
    pass

results = retrieve_top_k(query, documents, k=3)
results

In [ ]:
assert isinstance(results, list)
assert len(results) == 3
assert isinstance(results[0], tuple)
top_doc = results[0][0]
assert "language" in top_doc.lower() or "attention" in top_doc.lower()
scores = [score for _, score in results]
assert scores == sorted(scores, reverse=True)
print("All checks passed.")


# Блок 2 — Transformers

В этом блоке:
- работа с tokenizer,
- attention masks,
- sentence embeddings,
- inference,
- mini fine-tuning.



## Задание 4 — Zero-shot classification (1 балл)

`Zero-shot classification pipeline` позволяет классифицировать текст без дополнительного обучения. Такой подход был рассмотрен на семинаре, посвященном `transformers`

### Задание

Используйте:
```python
pipeline("zero-shot-classification")
```

Классы:
- technology
- sports
- finance
- politics

Необходимо:

* выполнить inference,
* определить наиболее вероятную тему новости,
* проанализировать confidence модели.

In [ ]:
candidate_labels = [
    "technology",
    "sports",
    "finance",
    "politics"
]

text = "Apple released a new AI chip for large-scale neural network training."

# Your code here

classifier = None
result = None
print(result)

In [ ]:
assert "labels" in result
assert len(result["labels"]) == 4
assert result["labels"][0] == "technology"
print("All checks passed.")


## Задание 5 — Tokenizer internals и embeddings (2 балла)

Transformer-модели не работают напрямую со строками.

Перед подачей текста в модель необходимо:

* разбить текст на токены,
* преобразовать токены в числовые id,
* сформировать attention mask,
* выровнять длины последовательностей.

За это отвечает `AutoTokenizer`.

`attention_mask` показывает модели:

* какие токены являются реальными,
* какие являются padding.

После прохождения текста через модель мы получаем скрытые представления (`hidden states`).

`CLS embedding` — это векторное представление всего предложения/текста, получаемое из специального токена `[CLS]` в Transformer-моделях семейства BERT.

### Часть A

Загрузите:
- `AutoTokenizer`
- `AutoModel`

Модель:
`distilbert-base-uncased`

Получите:
- `input_ids`
- `attention_mask`

### Часть B

Получите `CLS embeddings`.

Дополнительно:
- сравните предложения,
- вычислите cosine similarity matrix.


In [ ]:
sentences = [
    "Transformers are powerful models.",
    "Large language models process text.",
    "I enjoy playing basketball."
]

model_name = None

# Your code here

tokenizer = None
model = None

# Your code here

encoded = None
with torch.no_grad():
    outputs = None

cls_embeddings = None
similarity_matrix = None
print(similarity_matrix)

In [ ]:
assert encoded["input_ids"].shape == encoded["attention_mask"].shape
assert cls_embeddings.shape[0] == 3
assert cls_embeddings.shape[1] > 100
assert similarity_matrix.shape == (3, 3)
assert np.allclose(
    np.diag(similarity_matrix),
    1.0
)
assert similarity_matrix[0, 1] > similarity_matrix[0, 2]
print("All checks passed.")


## Задание 6 — Mini fine-tuning Transformers (2 балла)

`Fine-tuning` — один из ключевых подходов в современных NLP-системах.

Идея заключается в том, что:

1. модель предварительно обучается на огромном корпусе текстов (**pretraining**),
2. затем адаптируется под конкретную **downstream** задачу.

В этом задании вы выполните **mini fine-tuning Transformer** - модели для классификации новостей.

Вы будете использовать:

* `AutoModelForSequenceClassification`
* `Trainer`
* `TrainingArguments`

Fine-tuning включает:

* токенизацию,
* batching,
* обучение,
* inference,
* оценку качества.

### Часть A

1. Загрузите:

```python
AutoModelForSequenceClassification
```

2. Используйте `checkpoint`:

```python
distilbert-base-uncased
```

3. Выполните базовый `mini fine-tuning`. Получите `accuracy` $\geq 0.79$

### Часть B

Выполните mini fine-tuning с параметрами:
- Подберите гиперпараметры `epochs`, `batch_size` и размер `train subset` так, чтобы `accuracy` $\geq 0.85$
- Можно настроить и другие гиперпараметры

После обучения:
- получите predictions,
- вычислите `accuracy`.


**Обратите внимание**:

* даже короткое fine-tuning обычно даёт разумное качество,
* Transformer-модели используют contextual embeddings,
* модель учитывает порядок и контекст слов значительно лучше TF-IDF.

In [ ]:
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)


# Your code here

In [ ]:
assert model.config.num_labels == 4
assert "input_ids" in tokenized_train.features
assert len(pred_labels) == len(tokenized_test)
assert acc >= .85
print("All checks passed.")